# Comparaci?n nota final: XGBoost vs CatBoost
Analiza `uninion_enero_resultados.csv` y compara qu? modelo predice m?s cerca de la nota real,
filtrando solo filas con `Retiro_Asignatura_Cat == 0`.


In [1]:
import numpy as np
import pandas as pd

# Configura rutas/columnas
#input_csv = r'Resultados_Modelo_Excel_MegaPipeline\resultados_asig_prereq_mp_corr_cat2026-01-28110609.csv'
#input_csv= r"Resultados_Modelo_v2\resultados_asig_xg_cat2026-01-29120929.csv"
#input_csv= r"Resultados_Modelo_Excel_MegaPipeline\union_febrero_resultados.csv"
input_csv= r"Resultados_Modelo_v2/resultados_asig_xg_cat2026-02-23171813.csv"
print(input_csv)
sep = ';'

col_real_retiro = 'Retiro_Asignatura_Cat'
col_nota_real = 'resultado_final'
col_nota_pred_cat = 'Prediccion_CAT'
col_nota_pred_xgb = 'Prediccion_XGB'
col_nota_pred_rf= ''

def to_float(s):
    """Convierte notas con separador decimal ',' o '.' a float; no parseable -> NaN."""
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors='coerce')
    s_str = s.astype(str).str.strip()
    has_comma = s_str.str.contains(',', regex=False)
    has_dot = s_str.str.contains('.', regex=False)
    s_norm = s_str.copy()
    both = has_comma & has_dot
    s_norm = s_norm.where(~both, s_norm.str.replace('.', '', regex=False).str.replace(',', '.', regex=False))
    only_comma = has_comma & ~has_dot
    s_norm = s_norm.where(~only_comma, s_norm.str.replace(',', '.', regex=False))
    s_norm = s_norm.str.replace(' ', '', regex=False)
    return pd.to_numeric(s_norm, errors='coerce')

def compute_metrics(y, yhat):
    err = yhat - y
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err**2)))
    medae = float(np.median(np.abs(err)))
    return mae, rmse, medae

df = pd.read_csv(input_csv, sep=sep)

real_ret = pd.to_numeric(df[col_real_retiro], errors='coerce')
nota_real = to_float(df[col_nota_real])
nota_cat = to_float(df[col_nota_pred_cat])
nota_xgb = to_float(df[col_nota_pred_xgb])

mask = (real_ret == 0)
mask_base = mask & nota_real.notna()

# M?tricas por modelo (solo donde exista predicci?n)
m_cat = mask_base & nota_cat.notna()
m_xgb = mask_base & nota_xgb.notna()

y_cat = nota_real[m_cat]
yhat_cat = nota_cat[m_cat]
y_xgb = nota_real[m_xgb]
yhat_xgb = nota_xgb[m_xgb]

print('Filas totales:', len(df))
print('Filas retiro==0:', int(mask.sum()))
print('Filas con nota real:', int(mask_base.sum()))

if len(y_cat) > 0:
    mae, rmse, medae = compute_metrics(y_cat, yhat_cat)
    print(f'CatBoost -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MedAE: {medae:.4f} (n={len(y_cat)})')
else:
    print('CatBoost -> sin filas v?lidas')

if len(y_xgb) > 0:
    mae, rmse, medae = compute_metrics(y_xgb, yhat_xgb)
    print(f'XGBoost -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MedAE: {medae:.4f} (n={len(y_xgb)})')
else:
    print('XGBoost -> sin filas v?lidas')

# Comparaci?n directa (ambos disponibles)
mask_both = mask_base & nota_cat.notna() & nota_xgb.notna()
if mask_both.any():
    y = nota_real[mask_both]
    cat = nota_cat[mask_both]
    xgb = nota_xgb[mask_both]
    err_cat = np.abs(cat - y)
    err_xgb = np.abs(xgb - y)
    win_cat = int((err_cat < err_xgb).sum())
    win_xgb = int((err_xgb < err_cat).sum())
    tie = int((err_cat == err_xgb).sum())
    print('Comparaci?n fila a fila (ambos disponibles):')
    print('  Gana CatBoost:', win_cat)
    print('  Gana XGBoost:', win_xgb)
    print('  Empates:', tie)
else:
    print('No hay filas con ambas predicciones disponibles')


Resultados_Modelo_v2/resultados_asig_xg_cat2026-02-23171813.csv


C:\Users\00412\AppData\Local\Temp\ipykernel_31088\1789736887.py:40: DtypeWarning: Columns (41,42,45,47,48,51,52,54,56,58,61,63,65,66,68,70,72,74,76,78,81,83,84,87,89,90,92,95,97,99,100,103,104,113,115,119,120,122,124,127,128,131,133,134,136,138,140,142,144,148,150,152,154,156,160,163,164,168,171,172,174,177,181,183,184,191,193,195,197,198,200,202,205,206,209,210,213,214,218,220,223,224,226,229,230,232,234,237,238,240,243,245,247) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv, sep=sep)


Filas totales: 21607
Filas retiro==0: 20450
Filas con nota real: 20450
CatBoost -> MAE: 0.4700, RMSE: 0.6349, MedAE: 0.3700 (n=20450)
XGBoost -> MAE: 0.4567, RMSE: 0.6266, MedAE: 0.3500 (n=20450)
Comparaci?n fila a fila (ambos disponibles):
  Gana CatBoost: 9796
  Gana XGBoost: 10339
  Empates: 315
